In [31]:
#1 Instalar bibliotecas

"""
Usado para organizar, estruturar e analisar as notícias capturadas em tabelas
(DataFrames), facilitando filtros por data, agrupamentos e cálculos de estatísticas.
"""
!pip install pandas

#  conectar e ler os feeds RSS dos portais de notícias, extraindo automaticamente os títulos e informações de cada publicação
!pip install feedparser

#Fornece o modelo de inteligência artificial pré-treinado usado para classificar e analisar se o sentimento de cada notícia
!pip install transformers

"""
Funciona nos bastidores como um assistente do transformers, traduzindo e quebrando o texto das notícias em pedaços
menores (tokens) para que o modelo de IA possa processá-los corretamente
"""
!pip install sentencepiece

"""
Utilizado para construir e exibir os gráficos interativos que mostram visualmente a quantidade de menções e a
distribuição dos sentimentos de cada político
"""
!pip install plotly

In [32]:
# 2 = Instala funcionalidades das bibliotecas para usar

import pandas as pd
import feedparser
import plotly.express as px #criar gráficos interativos e visuais (como gráficos de barras de sentimentos ou menções)

from transformers import pipeline



In [33]:
#3 - Busca RSS FEEDS dos veículos

RSS_FEEDS = {

    "BAND":
    "https://rss.band.com.br/",

    "G1":
    "https://g1.globo.com/rss/g1/",

    "BRASIL PARALELO":
    "https://www.brasilparalelo.com.br/noticias",

    "SBT":
    "https://sbtnews.sbt.com.br/rss.xml",

    "CNN Brasil":
    "https://www.cnnbrasil.com.br/feed/",

    "InfoMoney":
    "https://www.infomoney.com.br/feed/",

    "InvestNews":
    "https://investnews.com.br/feed/",

    "Jovem Pan":
    "https://jovempan.com.br/feed/",

    "Revista Oeste":
    "https://revistaoeste.com/feed/",

    "O Antagonista":
    "https://www.oantagonista.com/feed/",

    "Brazil Journal":
    "https://braziljournal.com/feed/",

    "Gazeta do Povo":
    "https://www.gazetadopovo.com.br/feed/rss/ultimas-noticias.xml",

    "The Economist":
    "https://www.economist.com/latest/rss.xml",

    "CartaCapital":
    "https://www.cartacapital.com.br/feed/",

    "Poder360":
    "https://www.poder360.com.br/feed/",

    "Terra":
    "https://www.terra.com.br/rss/Controller?channelid=20e07ef2795b2310VgnVCM3000009af154d0RCRD&ctName=atomo-noticia",

    "CBN":
    "https://cbn.globo.com/rss/cbn/",

  # RURAL

    "CNA":
    "https://cnabrasil.org.br/feed",

    "Agro Goiás":
    "https://agrogoias.com.br/feed/",

    "Presente Rural":
    "https://opresenterural.com.br/feed/",

    "Parana Cooperativo":
    "https://paranacooperativo.coop.br/feed",

    "Globo Rural":
    "https://revistagloborural.globo.com/rss.xml",

    "Agrolink":
    "https://www.agrolink.com.br/rss"


}


Overwriting monitor_de_noticias/src/feeds.py


In [34]:
# 4 - Dicionário de SELEÇÃO DE TEMAS

TEMAS = {

    "Economia": [
        "economia",
        "mercado",
        "pib",
        "inflação",
        "juros",
        "dólar",
        "emprego",
        "trabalho",
        "desemprego",
        "renda",
        "exportações"
    ],

    "Impostos": [
        "imposto",
        "tributária",
        "taxa",
        "tributo"
    ],

    "Segurança": [
        "crime",
        "violência",
        "polícia",
        "segurança",
        "feminicídios",
        "corrupção"
    ],

    "STF": [
        "stf",
        "supremo",
        "moraes"
    ],

    "Eleições": [
        "eleição",
        "urna",
        "voto",
        "tse",
        "fake news",
        "compra de votos",
        "dinheiro"
    ],

    "Programas Sociais": [
        "bolsa família",
        "auxílio",
        "benefício"
    ],

    "Relações Exteriores": [
        "china",
        "eua",
        "brics",
        "onu",
        "trump"
    ]
}

print(type(TEMAS))

Overwriting monitor_de_noticias/src/temas.py


In [35]:
#5 - IMPORTA MODELO DE ANÁLISE DE SENTIMENTO

from transformers import pipeline

modelo = pipeline(
    "text-classification",
    model="cardiffnlp/twitter-xlm-roberta-base-sentiment",
    tokenizer="cardiffnlp/twitter-xlm-roberta-base-sentiment"
)

def analisar_sentimento(texto):

    resultado = modelo(texto[:512])[0] #Limita o texto da notícia aos primeiros 512 caracteres

    """ [0]: O modelo devolve os resultados dentro de uma lista (por exemplo,
    [{'label': 'negative', 'score': 0.95}]). Esse [0] serve para pegar diretamente
    o primeiro (e único) resultado dessa lista, facilitando o acesso às informações
    de sentimento (label) e certeza (score) nas linhas seguintes.
    """

    label = resultado["label"]
    score = resultado["score"]

    if label in ["LABEL_2", "positive"]:
        sentimento = "POSITIVO"

    elif label in ["LABEL_0", "negative"]:
        sentimento = "NEGATIVO"

    else:
        sentimento = "NEUTRO"

    return sentimento, score

print("Modelo carregado!")
print(type(modelo))

Overwriting monitor_de_noticias/src/sentimento.py


In [36]:
#6 - SELEÇÃO DE TERMOS A SEREM ANALISADOS NOS TÍTULOS

LULA_TERMS = [
    "lula",
    "luiz inácio",
    "presidente lula",
    "governo lula",
    "pt"
]

BOLSONARO_TERMS = [
    "flavio bolsonaro",
    "bolsonaro",
    "governo bolsonaro",
    "ex-presidente bolsonaro",
    "pl"
]


def identificar_candidato(texto):

    texto = str(texto).lower()

    candidatos = []

    if any(
        termo in texto
        for termo in [
            "lula",
            "luiz inácio",
            "presidente lula"
        ]
    ):
        candidatos.append("Lula")

    if any(
        termo in texto
        for termo in [
            "flávio bolsonaro",
            "flavio"
            "bolsonaro"
        ]
    ):
        candidatos.append("Flávio Bolsonaro")

    if any(
        termo in texto
        for termo in [
            "jair bolsonaro",
            "presidente bolsonaro"
        ]
    ):
        candidatos.append("Jair Bolsonaro")

    if any(
        termo in texto
        for termo in [
            "eduardo bolsonaro"
        ]
    ):
        candidatos.append("Eduardo Bolsonaro")

    if any(
        termo in texto
        for termo in [
            "carlos bolsonaro"
        ]
    ):
        candidatos.append("Carlos Bolsonaro")

    return candidatos


def identificar_tema(texto):

    texto = texto.lower()

    for tema, palavras in TEMAS.items():

        for palavra in palavras:

            if palavra in texto:
                return tema

    return "Outros"

    import unicodedata

 # NORMALIZAR ACENTO

def normalizar(texto):

    texto = texto.lower()

    texto = unicodedata.normalize(
        "NFD",
        texto
    )

    return "".join(
        c for c in texto
        if unicodedata.category(c) != "Mn"
    )

# ANALISAR SENTIMENTO

def analisar_sentimento(texto):

    try:

        resultado = modelo(texto[:512])[0]

        label = resultado["label"]

        score = resultado["score"]

        if label in ["positive", "LABEL_2"]:
            sentimento = "POSITIVO"

        elif label in ["negative", "LABEL_0"]:
            sentimento = "NEGATIVO"

        else:
            sentimento = "NEUTRO"

        return sentimento, score

    except:

        return "NEUTRO", 0

Overwriting monitor_de_noticias/src/candidatos.py


In [37]:
#7 - CRIA DATA FRAME

dados = []

for fonte, url in RSS_FEEDS.items():

    print("Lendo:", fonte)

    try:

        feed = feedparser.parse(url)

        for item in feed.entries:

            titulo = item.title

            candidatos = identificar_candidato(titulo)

            if not candidatos:
                continue

            sentimento, confianca = analisar_sentimento(
                titulo
            )

            tema = identificar_tema(
                titulo
            )

            for candidato in candidatos:

                dados.append({

                    "fonte": fonte,

                    "titulo": titulo,

                    "candidato": candidato,

                    "tema": tema,

                    "sentimento": sentimento,

                    "data": pd.to_datetime(
                        item.published,
                        utc=True,
                        errors="coerce"
                    ),

                    "confianca": round(
                        confianca,
                        3
                    )

                })

    except Exception as e:

        print(
            f"Erro em {fonte}:",
            e
        )

df = pd.DataFrame(dados)

# Converter para datetime
df["data"] = pd.to_datetime(
    df["data"],
    utc=True,
    errors="coerce"
)

# Converter UTC → Brasília
df["data"] = df["data"].dt.tz_convert(
    "America/Sao_Paulo"
)

# Data/hora atual
agora = pd.Timestamp.now(
    tz="America/Sao_Paulo"
)

# Filtrar apenas últimas 72 horas
df = df[
    df["data"] >= (
        agora - pd.Timedelta(hours=128)
    )
]

# Ordenar por data
df = df.sort_values(
    by="data",
    ascending=False
)

# Data formatada para exibição
df["data"] = df["data"].dt.strftime(
    "%d/%m/%Y %H:%M:%S"
)

print(
    f"Notícias nas últimas 72h: {len(df)}"
)

df.head(500)

# Configura o Pandas para nunca cortar linha

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

display(df)

Overwriting monitor_de_noticias/src/coleta.py


In [38]:
#8 CRIA RANKING POR CANDIDATO, QUANTIDADE DE NOTÍCIAS E SENTIMENTO. ESTIPULA NÍVEL DE CONFIABILIDADE DA CLASSIFICAÇÃO ( SENTIMENTO: POSITIVO, NEGATIVO OU NEUTRO )


dados = []

for fonte, url in RSS_FEEDS.items():

    print("Lendo:", fonte)

    try:

        feed = feedparser.parse(url)

        for item in feed.entries:

            titulo = item.title

            candidatos = identificar_candidato(titulo)

            if not candidatos:
                continue

            sentimento, confianca = analisar_sentimento(
                titulo
            )

            tema = identificar_tema(
                titulo
            )

            for candidato in candidatos:

                dados.append({

                    "fonte": fonte,

                    "titulo": titulo,

                    "candidato": candidato,

                    "tema": tema,

                    "sentimento": sentimento,

                    "data": pd.to_datetime(
                        item.published,
                        utc=True,
                        errors="coerce"
                    ),

                    "confianca": round(
                        confianca,
                        3
                    )

                })

    except Exception as e:

        print(
            f"Erro em {fonte}:",
            e
        )

df = pd.DataFrame(dados)

# Converter para datetime
df["data"] = pd.to_datetime(
    df["data"],
    utc=True,
    errors="coerce"
)

# Converter UTC → Brasília
df["data"] = df["data"].dt.tz_convert(
    "America/Sao_Paulo"
)

# Data/hora atual
agora = pd.Timestamp.now(
    tz="America/Sao_Paulo"
)

# Filtrar apenas últimas 72 horas
df = df[
    df["data"] >= (
        agora - pd.Timedelta(hours=72)
    )
]

# Ordenar por data
df = df.sort_values(
    by="data",
    ascending=False
)

# Data formatada para exibição
df["data"] = df["data"].dt.strftime(
    "%d/%m/%Y %H:%M:%S"
)

print(
    f"Notícias nas últimas 72h: {len(df)}"
)

df.head(500)

print("\n")
print("="*80)
print("RANKING POSITIVAS X NEGATIVAS")
print("="*80)

ranking_sentimento = (

    df

    .groupby(
        ["fonte", "candidato"]
    )

    .agg(

        total_postagens=(
            "titulo",
            "count"
        ),

        positivas=(
            "sentimento",
            lambda x:
            (x == "POSITIVO").sum()
        ),

        negativas=(
            "sentimento",
            lambda x:
            (x == "NEGATIVO").sum()
        ),

        neutras=(
            "sentimento",
            lambda x:
            (x == "NEUTRO").sum()
        ),

        confianca_media=(
            "confianca",
            "mean"
        ),

        ultima_publicacao=(
            "data",
            "max"
        )

    )

    .reset_index()

)

ranking_sentimento["saldo"] = (
    ranking_sentimento["positivas"]
    -
    ranking_sentimento["negativas"]
)

ranking_sentimento["confianca_media"] = (
    ranking_sentimento["confianca_media"]
    .round(3)
)

ranking_sentimento = ranking_sentimento.sort_values(
    by="total_postagens",
    ascending=False
)

display(ranking_sentimento)

Overwriting monitor_de_noticias/src/ranking.py


In [39]:

"""
# Monitor de Notícias

Sistema de monitoramento automatizado de notícias com coleta em múltiplos portais de mídia via RSS, identificação de temas, monitoramento de menções a atores políticos e análise de sentimento utilizando Inteligência Artificial.

O projeto captura manchetes publicadas por veículos de comunicação previamente definidos, classifica o conteúdo por tema, identifica candidatos monitorados e calcula automaticamente o sentimento predominante das notícias.

---

## Objetivo

Criar uma ferramenta capaz de:

- Coletar notícias em tempo real através de feeds RSS.
- Monitorar menções a nomes e grupos políticos.
- Classificar notícias por tema de interesse.
- Avaliar o sentimento das manchetes utilizando IA.
- Medir volume de exposição na mídia.
- Gerar rankings por veículo, candidato e sentimento.
- Produzir indicadores de reputação e cobertura jornalística.

---

## Funcionalidades

### Coleta Automatizada

O sistema consulta periodicamente feeds RSS de diversos veículos de comunicação, incluindo:

- G1
- Band
- CNN Brasil
- Jovem Pan
- SBT News
- Poder360
- CartaCapital
- Brasil Paralelo
- Gazeta do Povo
- Revista Oeste
- O Antagonista
- Terra
- CBN
- The Economist
- InfoMoney
- InvestNews
- Brazil Journal

Além de fontes especializadas do setor agropecuário:

- CNA
- Agro Goiás
- Globo Rural
- Agrolink
- Presente Rural
- Paraná Cooperativo

---

### Identificação de Candidatos

O monitor identifica automaticamente citações relacionadas a:

- Luiz Inácio Lula da Silva
- Jair Bolsonaro
- Flávio Bolsonaro
- Eduardo Bolsonaro
- Carlos Bolsonaro

A lógica pode ser facilmente expandida para incluir novos nomes.

---

### Classificação por Temas

As notícias são agrupadas em categorias estratégicas:

- Economia
- Impostos
- Segurança
- STF
- Eleições
- Programas Sociais
- Relações Exteriores
- Outros

A classificação ocorre por meio de palavras-chave associadas a cada tema.

---

### Análise de Sentimento

O sistema utiliza o modelo:

`cardiffnlp/twitter-xlm-roberta-base-sentiment`

para classificar cada manchete em:

- POSITIVO
- NEGATIVO
- NEUTRO

Além da classificação, o modelo retorna um índice de confiança para cada análise.

---

## Tecnologias Utilizadas

### Python

Linguagem principal do projeto.

### Pandas

Utilizado para:

- Estruturação dos dados
- Tratamento de tabelas
- Filtros
- Agrupamentos
- Estatísticas

### Feedparser

Responsável pela leitura dos feeds RSS e extração das publicações.

### Transformers

Biblioteca da Hugging Face utilizada para execução do modelo de IA.

### SentencePiece

Utilizado pelo modelo de linguagem para tokenização dos textos.

### Plotly

Biblioteca destinada à construção de gráficos e dashboards interativos.

---

## Instalação

### Clonar o repositório

```bash
git clone https://github.com/SEU-USUARIO/monitor_de_noticias.git
```

### Entrar na pasta

```bash
cd monitor_de_noticias
```

### Instalar dependências

```bash
pip install pandas
pip install feedparser
pip install transformers
pip install sentencepiece
pip install plotly
```

Ou:

```bash
pip install -r requirements.txt
```

### Abra o notebook

Abra o arquivo:

monitor_de_noticias.ipynb
no Google Colab ou Jupyter Notebook.
```

---

## Fluxo de Execução

### 1. Captura de Notícias

O sistema acessa todos os feeds RSS cadastrados.

### 2. Identificação de Menções

Cada manchete é analisada para identificar os candidatos monitorados.

### 3. Classificação por Temas

As notícias são categorizadas por assunto.

### 4. Análise de Sentimento

As manchetes são classificadas automaticamente em:

- Positivas
- Negativas
- Neutras

### 5. Consolidação

Os dados são armazenados em DataFrames Pandas.

### 6. Geração de Rankings

O sistema produz métricas como:

- Total de notícias
- Quantidade de menções
- Sentimento predominante
- Saldo positivo x negativo
- Confiança média das análises
- Última publicação identificada

---

## Indicadores Gerados

### Ranking de Exposição

Quantidade total de notícias por candidato.

### Ranking de Sentimento

Comparação entre:

- Positivas
- Negativas
- Neutras

### Saldo de Imagem

```text
Saldo = Notícias Positivas - Notícias Negativas
```

### Confiança Média

Média dos scores retornados pelo modelo de IA.

---

## Estrutura dos Dados

| Campo | Descrição |
|---------|---------|
| fonte | Veículo de comunicação |
| titulo | Título da notícia |
| candidato | Nome identificado |
| tema | Categoria temática |
| sentimento | Positivo, Negativo ou Neutro |
| data | Data da publicação |
| confianca | Score da IA |

---

## Possíveis Evoluções

- Dashboard em Power BI.
- Dashboard com Streamlit.
- Exportação automática para Excel.
- Geração de relatórios em PDF.
- Alertas por e-mail.
- Monitoramento de novos candidatos.
- Monitoramento de empresas, marcas e instituições.
- Armazenamento em banco de dados.
- Integração com APIs de notícias.

---

## Observações

A classificação é realizada sobre os títulos das notícias. O sentimento identificado representa a interpretação do modelo de IA sobre o texto analisado e não necessariamente o posicionamento editorial do veículo.





Writing monitor_de_noticias/README.md


In [42]:
!find monitor_de_noticias

monitor_de_noticias
monitor_de_noticias/dados
monitor_de_noticias/dados/processados
monitor_de_noticias/dados/brutos
monitor_de_noticias/requirements.txt
monitor_de_noticias/outputs
monitor_de_noticias/outputs/tabelas
monitor_de_noticias/outputs/graficos
monitor_de_noticias/src
monitor_de_noticias/src/coleta.py
monitor_de_noticias/src/ranking.py
monitor_de_noticias/src/feeds.py
monitor_de_noticias/src/temas.py
monitor_de_noticias/src/candidatos.py
monitor_de_noticias/src/sentimento.py
monitor_de_noticias/.ipynb_checkpoints
monitor_de_noticias/README.md
